# VeReMi-Graph v1.0 — PyTorch Geometric Converter and Validator

This notebook creates the graph-ML-ready distribution layer **from the public Parquet release**.

Input:
`RELEASE_ROOT / "data"`

Schemas:
`RELEASE_ROOT / "schemas"`

Output:
`RELEASE_ROOT / "pyg"`

One PyG `Data` object represents one 1-second graph snapshot.

Each object contains:
- `x` — node feature matrix;
- `edge_index` — directed sender→receiver topology;
- `edge_attr` — edge/message features;
- `edge_y_multiclass` — classes 0..5 corresponding to VeReMi labels 0,1,2,4,8,16;
- `edge_y_binary` — benign/malicious;
- availability masks for expected temporal-history nulls;
- previous-snapshot graph context;
- original `vehicle_id` values for traceability.

Expected first-history NaNs are converted to zero **only in tensor values**, while masks preserve whether a value was originally available. This introduces no validation/test-derived statistics.


In [ ]:
from pathlib import Path
import sys

# Locate repository root when notebook is executed from notebooks/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name in {"notebooks", "analysis", "examples"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib

try:
    import torch
    from torch_geometric.data import Data
except ImportError as e:
    raise ImportError(
        "Install PyTorch and PyTorch Geometric in the active environment before running this notebook."
    ) from e

from src.config import RELEASE_ROOT
ROOT = RELEASE_ROOT
NODE_DIR = ROOT/"data"/"nodes"
EDGE_DIR = ROOT/"data"/"edges"
SNAP_DIR = ROOT/"data"/"snapshots"
SCHEMA_DIR = ROOT/"schemas"
SPLIT_DIR = ROOT/"splits"
PYG_DIR = ROOT/"pyg"
PYG_DIR.mkdir(parents=True, exist_ok=True)

for s in ["train","validation","test"]:
    (PYG_DIR/s).mkdir(parents=True, exist_ok=True)

NODE_FEATURES = json.loads((SCHEMA_DIR/"node_input_features.json").read_text(encoding="utf-8"))
EDGE_FEATURES = json.loads((SCHEMA_DIR/"edge_input_features.json").read_text(encoding="utf-8"))
GRAPH_FEATURES = json.loads((SCHEMA_DIR/"graph_context_features.json").read_text(encoding="utf-8"))

manifest = pd.read_csv(SPLIT_DIR/"full_split_manifest.csv")

LABEL_MAP = {0:0, 1:1, 2:2, 4:3, 8:4, 16:5}

print("Node features:", len(NODE_FEATURES))
print("Edge features:", len(EDGE_FEATURES))
print("Graph-context features:", len(GRAPH_FEATURES))
print("Archives:", len(manifest))


In [ ]:
def matrix_and_mask(df, columns):
    m = df[columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    mask = np.isfinite(m)
    values = np.where(mask, m, 0.0).astype(np.float32)
    return values, mask.astype(np.bool_)


def build_graph(node_df, edge_df, snap_df, snapshot_id):
    nodes = node_df[node_df["snapshot_id"] == snapshot_id].copy()
    edges = edge_df[edge_df["snapshot_id"] == snapshot_id].copy()
    graph_row = snap_df[snap_df["snapshot_id"] == snapshot_id].copy()

    if graph_row.empty:
        raise ValueError(f"Missing snapshot row {snapshot_id}")

    node_ids = set(nodes["vehicle_id"].dropna().astype(int).tolist())
    node_ids |= set(edges["sender"].dropna().astype(int).tolist())
    node_ids |= set(edges["receiver"].dropna().astype(int).tolist())
    node_ids = sorted(node_ids)

    node_to_idx = {v:i for i,v in enumerate(node_ids)}

    # Align feature rows exactly to graph-node order.
    nodes = nodes.set_index("vehicle_id").reindex(node_ids).reset_index()

    x_np, x_mask_np = matrix_and_mask(nodes, NODE_FEATURES)

    if len(edges):
        src = edges["sender"].astype(int).map(node_to_idx).to_numpy(dtype=np.int64)
        dst = edges["receiver"].astype(int).map(node_to_idx).to_numpy(dtype=np.int64)
        edge_index = torch.tensor(np.vstack([src,dst]), dtype=torch.long)

        edge_attr_np, edge_mask_np = matrix_and_mask(edges, EDGE_FEATURES)

        raw_y = pd.to_numeric(edges["attackerType"], errors="raise").astype(int).to_numpy()
        y_multi = np.array([LABEL_MAP[int(v)] for v in raw_y], dtype=np.int64)
        y_binary = (raw_y != 0).astype(np.int64)
    else:
        edge_index = torch.empty((2,0), dtype=torch.long)
        edge_attr_np = np.empty((0,len(EDGE_FEATURES)), dtype=np.float32)
        edge_mask_np = np.empty((0,len(EDGE_FEATURES)), dtype=np.bool_)
        y_multi = np.empty((0,), dtype=np.int64)
        y_binary = np.empty((0,), dtype=np.int64)

    graph_np, graph_mask_np = matrix_and_mask(graph_row.iloc[[0]], GRAPH_FEATURES)

    data = Data(
        x=torch.tensor(x_np, dtype=torch.float32),
        edge_index=edge_index,
        edge_attr=torch.tensor(edge_attr_np, dtype=torch.float32)
    )

    data.edge_y_multiclass = torch.tensor(y_multi, dtype=torch.long)
    data.edge_y_binary = torch.tensor(y_binary, dtype=torch.long)
    data.node_history_mask = torch.tensor(x_mask_np, dtype=torch.bool)
    data.edge_history_mask = torch.tensor(edge_mask_np, dtype=torch.bool)
    data.graph_context = torch.tensor(graph_np[0], dtype=torch.float32)
    data.graph_context_mask = torch.tensor(graph_mask_np[0], dtype=torch.bool)
    data.vehicle_id = torch.tensor(node_ids, dtype=torch.long)
    data.snapshot_id = torch.tensor([int(snapshot_id)], dtype=torch.long)

    return data


## Dry run on one graph


In [ ]:
sample_archive = manifest.iloc[0]["archive_name"]

n = pd.read_parquet(NODE_DIR/f"{sample_archive}.parquet")
e = pd.read_parquet(EDGE_DIR/f"{sample_archive}.parquet")
s = pd.read_parquet(SNAP_DIR/f"{sample_archive}.parquet")

g = build_graph(n,e,s,0)

print(sample_archive)
print(g)
print("x:", tuple(g.x.shape))
print("edge_index:", tuple(g.edge_index.shape))
print("edge_attr:", tuple(g.edge_attr.shape))
print("edge_y_multiclass:", tuple(g.edge_y_multiclass.shape))
print("graph_context:", tuple(g.graph_context.shape))


## Convert all 22,500 temporal snapshots


In [ ]:
records = []
failures = []

for i,row in manifest.iterrows():
    archive = row["archive_name"]
    split = row["split"]

    print(f"[{i+1}/{len(manifest)}] {archive} -> {split}")

    try:
        n = pd.read_parquet(NODE_DIR/f"{archive}.parquet")
        e = pd.read_parquet(EDGE_DIR/f"{archive}.parquet")
        s = pd.read_parquet(SNAP_DIR/f"{archive}.parquet")

        out_dir = PYG_DIR/split/archive
        out_dir.mkdir(parents=True, exist_ok=True)

        for snapshot_id in range(100):
            g = build_graph(n,e,s,snapshot_id)
            p = out_dir/f"snapshot_{snapshot_id:03d}.pt"
            torch.save(g,p)

            records.append({
                "archive_name": archive,
                "split": split,
                "snapshot_id": snapshot_id,
                "node_count": int(g.x.shape[0]),
                "edge_count": int(g.edge_attr.shape[0]),
                "node_feature_count": int(g.x.shape[1]),
                "edge_feature_count": int(g.edge_attr.shape[1]),
                "graph_context_feature_count": int(g.graph_context.shape[0]),
                "file_path": p.relative_to(ROOT).as_posix(),
                "file_size_bytes": p.stat().st_size
            })

    except Exception as ex:
        failures.append({"archive_name":archive,"split":split,"error":repr(ex)})
        print("FAILED:",repr(ex))

conversion = pd.DataFrame(records)
failure_df = pd.DataFrame(failures)

conversion.to_csv(PYG_DIR/"pyg_snapshot_manifest.csv",index=False)
failure_df.to_csv(PYG_DIR/"pyg_conversion_failures.csv",index=False)

print("\nPyG snapshots:",len(conversion))
print("Failed archives:",len(failure_df))


## Validate counts and frozen dimensions


In [ ]:
summary = pd.DataFrame({
    "metric": [
        "snapshot_objects",
        "failed_archives",
        "total_nodes_across_snapshots",
        "total_edges",
        "node_feature_dimension",
        "edge_feature_dimension",
        "graph_context_dimension"
    ],
    "value": [
        len(conversion),
        len(failure_df),
        int(conversion.node_count.sum()),
        int(conversion.edge_count.sum()),
        int(conversion.node_feature_count.iloc[0]),
        int(conversion.edge_feature_count.iloc[0]),
        int(conversion.graph_context_feature_count.iloc[0])
    ]
})
display(summary)

assert len(conversion) == 22_500
assert len(failure_df) == 0
assert int(conversion.edge_count.sum()) == 39_136_291
assert conversion.node_feature_count.nunique() == 1
assert conversion.edge_feature_count.nunique() == 1
assert conversion.graph_context_feature_count.nunique() == 1
assert int(conversion.node_feature_count.iloc[0]) == len(NODE_FEATURES)
assert int(conversion.edge_feature_count.iloc[0]) == len(EDGE_FEATURES)
assert int(conversion.graph_context_feature_count.iloc[0]) == len(GRAPH_FEATURES)


## Deep sample validation across train, validation, and test


In [ ]:
deep = []

for split in ["train","validation","test"]:
    files = sorted((PYG_DIR/split).glob("*/*.pt"))
    idx = np.unique(np.linspace(0,len(files)-1,min(20,len(files)),dtype=int))

    for j in idx:
        p = files[j]
        g = torch.load(p, weights_only=False)

        deep.append({
            "split": split,
            "file": p.relative_to(ROOT).as_posix(),
            "x_nan": bool(torch.isnan(g.x).any().item()),
            "edge_attr_nan": bool(torch.isnan(g.edge_attr).any().item()),
            "graph_context_nan": bool(torch.isnan(g.graph_context).any().item()),
            "node_count": int(g.x.shape[0]),
            "edge_count": int(g.edge_attr.shape[0]),
            "edge_index_min": int(g.edge_index.min().item()) if g.edge_index.numel() else -1,
            "edge_index_max": int(g.edge_index.max().item()) if g.edge_index.numel() else -1,
            "label_min": int(g.edge_y_multiclass.min().item()) if g.edge_y_multiclass.numel() else -1,
            "label_max": int(g.edge_y_multiclass.max().item()) if g.edge_y_multiclass.numel() else -1
        })

deep_df = pd.DataFrame(deep)
deep_df.to_csv(PYG_DIR/"pyg_deep_validation_sample.csv",index=False)
display(deep_df.head())

assert not deep_df.x_nan.any()
assert not deep_df.edge_attr_nan.any()
assert not deep_df.graph_context_nan.any()
assert (deep_df.edge_index_max < deep_df.node_count).all()
assert (deep_df.label_min >= 0).all()
assert (deep_df.label_max <= 5).all()


## Checksums for PyG objects


In [ ]:
def sha256_file(path,chunk_size=8*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            chunk=f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

files = sorted(PYG_DIR.glob("*/*/*.pt"))
rows = []

for i,p in enumerate(files,start=1):
    rows.append({
        "relative_path":p.relative_to(ROOT).as_posix(),
        "size_bytes":p.stat().st_size,
        "sha256":sha256_file(p)
    })
    if i==1 or i%1000==0:
        print(f"Checksummed {i}/{len(files)}")

pyg_checksums = pd.DataFrame(rows)
pyg_checksums.to_csv(ROOT/"checksums"/"pyg_sha256.csv",index=False)

assert len(pyg_checksums)==22_500
print("PyG checksums:",len(pyg_checksums))


## Final PyG release report


In [ ]:
checks = [
    ("22,500 PyG graphs", len(conversion)==22_500),
    ("No conversion failures", len(failure_df)==0),
    ("39,136,291 edges represented", int(conversion.edge_count.sum())==39_136_291),
    ("Node dimension frozen", int(conversion.node_feature_count.iloc[0])==len(NODE_FEATURES)),
    ("Edge dimension frozen", int(conversion.edge_feature_count.iloc[0])==len(EDGE_FEATURES)),
    ("Graph context dimension frozen", int(conversion.graph_context_feature_count.iloc[0])==len(GRAPH_FEATURES)),
    ("No NaNs in deep tensor sample", not (deep_df.x_nan.any() or deep_df.edge_attr_nan.any() or deep_df.graph_context_nan.any())),
    ("22,500 PyG checksums", len(pyg_checksums)==22_500)
]

report = pd.DataFrame(checks,columns=["check","passed"])
report["status"] = np.where(report.passed,"PASS","FAIL")
overall = "PASS_PYG_RELEASE" if report.passed.all() else "REVIEW_REQUIRED"

report.loc[len(report)] = ["FINAL_PYG_RELEASE_STATUS", report.passed.all(), overall]
report.to_csv(PYG_DIR/"PYG_RELEASE_REPORT.csv",index=False)

display(report)
print("\nFinal status:",overall)
print("PyG root:",PYG_DIR)


## Modeling note

These are **raw release graphs**, not scaled model inputs.

Any learned normalization/scaling must be fitted on the 157 training archives only and then applied unchanged to validation and test. The public PyG layer therefore remains reusable independently of any single model.
